# **Langchain with Deepagents**

In [1]:
import os
from daytona import Daytona
from langchain_anthropic import ChatAnthropic
from langchain_daytona import DaytonaSandbox
from deepagents import create_deep_agent
from dotenv import load_dotenv

load_dotenv()

def run_deep_agent_in_daytona():
    daytona = Daytona()

    # Connect to your existing sandbox shown in the dashboard ("my-daytona-sandbox")
    sandbox = daytona.get("my-daytona-sandbox")

    # Ensure it is started (in case it auto-stopped after idling)
    if sandbox.state != "started":
        sandbox.start()

    # 1. Use a relative path ("sales.csv") -> resolves to /home/daytona/sales.csv
    sample_csv = (
        "region,month,revenue,expenses\n"
        "East,Jan,12000,8000\n"
        "East,Feb,15500,8500\n"
        "West,Jan,9800,7200\n"
        "West,Feb,14200,7900\n"
    )
    sandbox.fs.upload_file(sample_csv.encode("utf-8"), "sales.csv")

    # 2. Wrap the sandbox as a Deep Agents backend
    backend = DaytonaSandbox(sandbox=sandbox)

    # 3. Create the Deep Agent (instruct it to work inside /home/daytona)
    agent = create_deep_agent(
        model=ChatAnthropic(model="claude-sonnet-4-6"),
        system_prompt=(
            "You are an autonomous data engineering and Python coding assistant. "
            "Your working directory is /home/daytona. "
            "Always plan your steps, write clean scripts, verify outputs by running them, "
            "and save final deliverables inside /home/daytona/output/."
        ),
        backend=backend,
    )

    # 4. Run the task using /home/daytona paths
    user_task = (
        "Analyze /home/daytona/sales.csv. Write a Python script to compute net profit "
        "and month-over-month growth per region, verify the script runs without errors, "
        "and save a clean Markdown summary to /home/daytona/output/summary.md."
    )

    print("Running Deep Agent in Daytona sandbox...\n")
    result = agent.invoke(
        {"messages": [{"role": "user", "content": user_task}]}
    )

    print("=== Agent Final Response ===")
    print(result["messages"][-1].content)

    # 5. Download using relative path ("output/summary.md")
    os.makedirs("./local_output", exist_ok=True)
    report_bytes = sandbox.fs.download_file("output/summary.md")
    with open("./local_output/summary.md", "wb") as f:
        f.write(report_bytes)

    print("\nSaved output/summary.md -> ./local_output/summary.md")


if __name__ == "__main__":
    run_deep_agent_in_daytona()



Running Deep Agent in Daytona sandbox...

=== Agent Final Response ===
Everything is perfect. Here's a full breakdown of what was done and what the results show:

---

## What the Script Does

The script (`/home/daytona/analyze_sales.py`) performs these steps:

| Step | Action |
|------|--------|
| **Load** | Reads `sales.csv` with Python's built-in `csv.DictReader` (no external dependencies) |
| **Net Profit** | Computes `net_profit = revenue − expenses` for every region/month row |
| **MoM Growth** | Calculates `(current − previous) / |previous| × 100%` within each region; first month is marked `N/A` |
| **Aggregates** | Rolls up totals and profit margins per region and overall |
| **Insights** | Auto-identifies best/worst regions and strongest MoM improvement |
| **Output** | Writes a clean, fully formatted Markdown file |

---

## Key Findings

### Overall KPIs
| Metric | Value |
|--------|-------|
| Total Revenue | $51,500 |
| Total Expenses | $31,600 |
| **Total Net Profit** | **